# 02 - Feature Engineering, Harmonization & Schema Mapping

## Overview
Heterogeneous electronic health records (EHR) and clinical trial registries use divergent column nomenclatures, measurements, and categorical encodings. 
This notebook demonstrates:
1. Canonical schema harmonization across diverse cohorts.
2. Clinical plausible range clipping & anomaly detection (e.g. systolic BP 60-250 mmHg).
3. Strategy for handling missing data without introducing lookahead bias.
4. Stratified cohort partitioning (`train.csv`, `val.csv`, `test.csv`).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

## 1. Inspecting the Canonical Clinical Schema
The standardized schema divides variables into clinical domains.

In [ ]:
from src.schema_mapping import CANONICAL_COLUMNS

print(f"Total Canonical Columns: {len(CANONICAL_COLUMNS)}")
print("\nCanonical features defined across clinical domains:")
for idx, col in enumerate(CANONICAL_COLUMNS, 1):
    print(f" {idx:2d}. {col}")

## 2. Plausibility & Physiological Outlier Bounds
In real-world health applications, biometric values out of physiological range (e.g., negative blood pressures or impossible heart rates) degrade ML reliability. 
Let's verify bounds filtering:

In [ ]:
PHYSIOLOGICAL_BOUNDS = {
    "age": (0, 120),
    "resting_bp": (50, 260),
    "max_heart_rate": (40, 240),
    "bmi": (10, 75),
    "cholesterol": (50, 600),
    "fasting_blood_sugar": (30, 500)
}

from src.schema_mapping import PROCESSED_DIR

clean_path = PROCESSED_DIR / "clean_combined_dataset.csv"
if clean_path.exists():
    df_clean = pd.read_csv(clean_path)
    print(f"Loaded clean dataset: {len(df_clean):,} records")
    
    print("\nClinical Bounds Check Summary:")
    for col, (low, high) in PHYSIOLOGICAL_BOUNDS.items():
        if col in df_clean.columns:
            series = df_clean[col].dropna()
            out_of_bounds = ((series < low) | (series > high)).sum()
            print(f"  * {col:<22}: Min={series.min():.1f}, Max={series.max():.1f} | Out of bounds: {out_of_bounds}")
else:
    print(f"Processed file not found at {clean_path}. Run build_training_data.py to generate.")

## 3. Preprocessing Pipeline (`ClinicalDataPreprocessor`)
`ClinicalDataPreprocessor` executes domain-aware imputation and transformation:
- Numeric features: Median imputation + standard scaling + missingness indicator flags.
- Categorical features: Most frequent imputation + one-hot encoding.
- History / Lifestyle flags: Binary casting with default 0 (absence of recorded symptom).

In [ ]:
from src.preprocessing import ClinicalDataPreprocessor

preprocessor = ClinicalDataPreprocessor()
print("Preprocessor initialized.")
print(f"Numeric features   : {len(preprocessor.numeric_features)} columns")
print(f"Categorical features : {len(preprocessor.categorical_features)} columns")
print(f"Binary flags       : {len(preprocessor.binary_features)} columns")

## 4. Train / Validation / Test Stratification
To prevent data leakage and ensure fair representation across rare medical conditions and clinical sources, splitting is stratified jointly on `source_dataset` and `target_risk`.

In [ ]:
train_path = PROCESSED_DIR / "train.csv"
val_path = PROCESSED_DIR / "val.csv"
test_path = PROCESSED_DIR / "test.csv"

if train_path.exists() and val_path.exists() and test_path.exists():
    df_train = pd.read_csv(train_path)
    df_val = pd.read_csv(val_path)
    df_test = pd.read_csv(test_path)
    
    total = len(df_train) + len(df_val) + len(df_test)
    print("Dataset Split Distribution:")
    print(f"  * Train set      : {len(df_train):>7,} rows ({len(df_train)/total*100:.1f}%) | Positives: {(df_train['target_risk']==1).mean()*100:.2f}%")
    print(f"  * Validation set : {len(df_val):>7,} rows ({len(df_val)/total*100:.1f}%) | Positives: {(df_val['target_risk']==1).mean()*100:.2f}%")
    print(f"  * Test set       : {len(df_test):>7,} rows ({len(df_test)/total*100:.1f}%) | Positives: {(df_test['target_risk']==1).mean()*100:.2f}%")
else:
    print("Train/val/test files missing in data/processed/. Run src/build_training_data.py")

## 5. Dataset Shift Verification
A vital question in medical machine learning is: *Do distributions vary widely across hospital sources?*
Let's inspect age and resting BP across major datasets:

In [ ]:
if 'df_clean' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    top_sources = df_clean['source_dataset'].value_counts().head(5).index
    df_subset = df_clean[df_clean['source_dataset'].isin(top_sources)]
    
    df_subset.boxplot(column='age', by='source_dataset', ax=axes[0])
    axes[0].set_title('Age Distribution Across Cohorts', fontweight='bold')
    axes[0].set_xlabel('')
    axes[0].tick_params(axis='x', rotation=30)
    
    df_subset.boxplot(column='resting_bp', by='source_dataset', ax=axes[1])
    axes[1].set_title('Resting BP Across Cohorts', fontweight='bold')
    axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=30)
    
    plt.suptitle('')
    plt.tight_layout()
    plt.show()